<a href="https://colab.research.google.com/github/YoussifKhaled77/FlyrankAI-Internship/blob/main/work/notebooks/w01_research_question_v1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YoussifKhaled77/FlyrankAI-Internship/blob/main/work/notebooks/w01_research_question_v1.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

**Lane: Lane 2 — Refresh / Content Opportunity Scoring.**

The question this lane answers: *which pages should a content reviewer look at first, given
limited review time?* I picked it over the other three predefined lanes for three reasons.

- **It matches the data I actually have this week.** The starter dataset
  (`data/raw/content_refresh_anonymized.csv`) and the runnable reference pipeline
  (`scripts/01`–`05`) are already built around exactly this question — a transparent baseline
  score, a trained model, and a ranked, reason-coded review queue. I can build a real,
  end-to-end version of this lane without the warehouse release.
- **It produces an action, not just an observation.** Lane 1 (Ranking Signal Analysis) and
  Lane 3 (Archetype Clustering) are valuable, but their outputs are reports and profiles — no
  one is handed a ranked list telling them what to look at Monday morning. Lane 2's output
  (a ranked queue with reason codes and a suggested action) is something a reviewer can use
  immediately.
- **The stakes are visible in the data itself**, as section 3 below shows with real numbers:
  a large share of the total search demand in this dataset sits on pages that are currently
  trending down. That is exactly the situation a prioritized review queue is for.

I considered Lane 4 (CTR/Engagement Opportunity Scoring) as well — it's a close cousin and its
signals (CTR-vs-position, engagement) show up as reason codes inside Lane 2's baseline anyway,
so I get most of its value without narrowing my target this early. I ruled out the freestyle
directions for week 1: AI Referral Opportunity is explicitly flagged as too sparse for anything
beyond light EDA, and Growth/Recovery/Momentum Prediction needs future-window daily data from
the warehouse release, which isn't part of this week's starter dataset.


In [ ]:
# Confirm the lane's data source is real and accessible before committing to it.
import pandas as pd

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

print(f"Rows x columns: {df.shape[0]:,} x {df.shape[1]}")
print(f"Distinct clients (client_id): {df['client_id'].nunique()}")
print(f"Distinct content items (content_id): {df['content_id'].nunique()}")


Rows x columns: 30,000 x 44
Distinct clients (client_id): 32
Distinct content items (content_id): 30000


## 2. The question: decision, action, cost of a wrong call

**Research question:** Which content pages should a reviewer check first this cycle, out of a
much larger inventory than anyone has time to review by hand?

**Decision it improves:** How a content/SEO reviewer with limited weekly capacity allocates
their attention — which handful of pages get looked at first, instead of reviewing pages in
whatever order they happen to appear, or not reviewing anything until traffic loss is obvious.

**Who acts, and what they do:** A content reviewer or SEO strategist (at FlyRank or at the
client) opens the ranked queue, reads the reason code(s) attached to each page near the top,
and takes one of a small set of actions per page — refresh the content, expand it, protect it
(it's fine, leave it), prune it, or simply keep monitoring it. If nobody would change what they
review first because of this output, the project has no real customer — so the queue has to be
something a reviewer would actually open and trust.

**Cost of a wrong call:**
- **False negative** (a genuinely at-risk page is *not* surfaced near the top): the page keeps
  losing visibility unnoticed. As section 3 shows, pages already trending down with real search
  demand carry roughly half of *all* impressions in this dataset — so missing the wrong page is
  not a minor miss, it's real traffic quietly eroding.
- **False positive** (a page is flagged urgent but isn't really a problem — e.g. a seasonal dip
  or a sibling page absorbing the traffic): the reviewer spends limited time on a page that
  didn't need it, which is itself a cost, because it pushes a real problem further down the
  queue.

Because review capacity is the actual bottleneck (a team can act on maybe the top 20–50
candidates in a cycle, not all 30,000 rows), the *ranking quality* of the queue — not just
whether a page is technically "flagged" — is what determines whether the right pages get seen.

**Why ML, not just a rule:** A simple rule (e.g. "flag anything trending down with decent
traffic") is a reasonable starting baseline, and I will build one first. But this repo's own
reference pipeline already shows the gap between a hand-written rule and a trained model on
this exact dataset: the rule baseline reaches Precision@50 ≈ 0.24, while a random forest reaches
≈ 0.74 — roughly three times as many of the top 50 flagged pages are correct. That gap exists
because the real signal is spread across many correlated, continuously-valued columns
(impressions, position, freshness, engagement, word count, and more) whose *combined* effect on
risk is too tangled to hand-write as a short list of if/else thresholds — exactly the situation
where a learned model earns its place over a plain rule.


In [ ]:
# Ground the "cost of a wrong call" claim in a real number: how much search demand
# currently sits on pages that look at-risk under a simple, explainable rule?
declining_with_demand = (df["trend_direction"] == "down") & (df["impressions_90d"] >= 100)

n_flagged = declining_with_demand.sum()
pct_flagged = declining_with_demand.mean() * 100
impressions_at_stake = df.loc[declining_with_demand, "impressions_90d"].sum()
impressions_total = df["impressions_90d"].sum()
pct_impressions_at_stake = 100 * impressions_at_stake / impressions_total

print(f"Pages trending down with real demand (impressions_90d >= 100): {n_flagged:,} "
      f"({pct_flagged:.1f}% of all rows)")
print(f"Share of ALL impressions_90d sitting on those pages: {pct_impressions_at_stake:.1f}%")


Pages trending down with real demand (impressions_90d >= 100): 13,152 (43.8% of all rows)
Share of ALL impressions_90d sitting on those pages: 51.2%


## 3. Quick look at the data (2-3 real numbers)

Three real numbers from `data/raw/content_refresh_anonymized.csv`, computed below:

1. **54.2% of pages (16,262 of 30,000) are currently labeled `trend_direction == "down"`.**
   That is the majority of the inventory — far too many for any reviewer to check by hand,
   which is the whole reason a *ranked* queue (not just a flag) is needed.
2. **43.8% of pages (13,152 of 30,000) are both trending down *and* have real search demand**
   (`impressions_90d >= 100`) — a simple, explainable rule for "worth a look." This is still
   too large a list for limited review capacity, which is exactly why ranking within it matters.
3. **Those 13,152 demand-backed, declining pages carry about 51% of all search impressions in
   the dataset.** Visibility loss isn't concentrated on obscure, low-traffic pages — roughly
   half of the measured search demand in this slice sits on pages that look at risk right now.

Together these numbers say: the problem is real (over half the inventory looks like it's
declining), it's too big to review by hand (13k+ candidates even after an obvious demand
filter), and the stakes are high (about half of all impressions are riding on that subset) —
which is exactly the situation a prioritized, evidence-backed review queue is built for.


In [ ]:
# The 2-3 headline statistics, computed directly from the real starter dataset.
trend_counts = df["trend_direction"].value_counts()
trend_pct = (df["trend_direction"].value_counts(normalize=True) * 100).round(1)

print("trend_direction counts:")
print(trend_counts)
print()
print("trend_direction as % of all rows:")
print(trend_pct)
print()
print(f"Stat 1 -> Pages trending down: {trend_counts['down']:,} "
      f"({trend_pct['down']}% of {len(df):,} rows)")
print(f"Stat 2 -> Declining AND impressions_90d >= 100: {n_flagged:,} "
      f"({pct_flagged:.1f}% of rows)")
print(f"Stat 3 -> Share of total impressions_90d on that group: {pct_impressions_at_stake:.1f}%")


trend_direction counts:
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64

trend_direction as % of all rows:
trend_direction
down      54.2
stable    19.9
up        14.6
new        7.5
flat       3.8
Name: proportion, dtype: float64

Stat 1 -> Pages trending down: 16,262 (54.2% of 30,000 rows)
Stat 2 -> Declining AND impressions_90d >= 100: 13,152 (43.8% of rows)
Stat 3 -> Share of total impressions_90d on that group: 51.2%


## 4. Careful words: what I can and can't claim

**What this project can claim, once it's built:**
- **Observed** patterns: which measured signals (impressions, position, freshness, engagement,
  and similar) tend to go together with a page currently trending down, in this anonymized
  90-day slice.
- **Directional** risk scores: a ranked ordering of which pages look more versus less likely to
  need attention, based on the patterns above — not a certainty for any single page.
- **Decision-support**, not decision-making: the output is a queue a human reviewer inspects and
  chooses from, with reason codes they can check against the real data — the model does not act
  on its own.

**What this project cannot claim, now or later:**
- **Not causal.** Nothing here shows that refreshing a page *causes* recovery, or that leaving
  a page alone *causes* further decline — that would need an experiment (e.g. before/after with
  a control group), which this dataset does not provide.
- **Not a Google algorithm claim.** I can describe correlations between observable signals and
  a trend label; I cannot claim to have found or predicted a Google ranking factor.
- **Not a guarantee.** "Trending down" is not the same as "will decline forever" — some of these
  pages are seasonal dips, some sit alongside a sibling page that absorbed the traffic
  (consolidation), and some are just noisy at low volume. The queue tells a reviewer where to
  look first, not what they will find.
- **The current label is a proxy, honestly.** `is_declining_label` is defined from
  `trend_direction`, which compares the last 30 days to the prior 30 — a bucket calculated from
  the *current* window, not a future outcome. It's a fine label to prove the workflow this week,
  but a stronger version of this lane (later weeks) should predict a *future* window instead, so
  the model is judged on what happens next rather than on how the present is currently labeled.


In [ ]:
# Sanity-check the leakage boundary I just described in words above:
# trend_direction / trend_pct define the label, so they must never be used as model features.
label_source_columns = ["trend_direction", "trend_pct"]
print("Label-source columns (label only, NEVER features):", label_source_columns)

# Also confirm content_id is a clean, unique identifier -- safe for grouping/joins,
# never as a feature, and never something to publish as if it were a real URL or client name.
print(f"content_id is unique per row: {df['content_id'].is_unique}")
print(f"client_id has {df['client_id'].nunique()} distinct pseudonymous values (grouping key only)")


Label-source columns (label only, NEVER features): ['trend_direction', 'trend_pct']
content_id is unique per row: True
client_id has 32 distinct pseudonymous values (grouping key only)


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
